In [4]:
from sklearn.model_selection import train_test_split,  GridSearchCV
from sklearn import dummy
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
from sklearn.dummy import DummyClassifier
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('default')
import pickle

# Importing Data

In [2]:
#importing the dataset
model_df = pd.read_csv('stigmatised_deprivations_and_loneliness.csv')

model_df_all_covs_df = model_df.loc[:, [
    # --- Demographic Features ---
    'Age',
    'Income',
    'Unemployed',
    'Not Working (Looking After Child or House)',
    'Not Working Sick or Disabled',
    'Living On my Own With Children Under 18',
    'Gender',
    # --- Health and Wellbeing Features ---
    'Illness Yes',
    # --- Financial Insecurity Features ---
    'Uses Foodbank',
    'Worry About Housing Payment',
    'Worry About Energy Payment',
    'Food Insecurity',
    'Energy Insecurity',

    
    # --- Social Support Features ---
    'Unable to Turn to Friends or Family',
    
    # --- MSOA/LSOA IMD Derived Features ---
    'IMD Score',
    'Living Environment Score',
    'Population Density',
    
    # --- Geographic and Infrastructure Features ---
    'Distance to Bus Stop',
    'Distance to Underground Access',
    'Distance to Railway Access',
    'Distance to Greenspace',
    
    # --- Loneliness Indicator ---
    'new_loneliness_binary'
]]
model_df_no_food_insecurity = model_df.loc[:, [
    # --- Demographic Features ---
    'Age',
    'Income',
    #'updated_income',
    'Unemployed',
    'Not Working (Looking After Child or House)',
    'Not Working Sick or Disabled',
    'Living On my Own With Children Under 18',
    'Gender',
    # --- Health and Wellbeing Features ---
    'Illness Yes',
    # --- Financial Insecurity Features ---
    'Uses Foodbank',
    'Worry About Housing Payment',
    'Worry About Energy Payment',
    #'Food Insecurity',
    'Energy Insecurity',
 
    # --- Social Support Features ---
    'Unable to Turn to Friends or Family',
    
    # --- MSOA/LSOA IMD Derived Features ---
    'IMD Score',
    'Living Environment Score',
    'Population Density',
    
    # --- Geographic and Infrastructure Features ---
    'Distance to Bus Stop',
    'Distance to Underground Access',
    'Distance to Railway Access',
    'Distance to Greenspace',
    
    # --- Loneliness Indicator ---
    'new_loneliness_binary'
]]
model_df_no_energy_insecurity = model_df.loc[:, [
    # --- Demographic Features ---
    'Age',
    'Income',
    'Unemployed',
    'Not Working (Looking After Child or House)',
    'Not Working Sick or Disabled',
    'Living On my Own With Children Under 18',
    'Gender',
    # --- Health and Wellbeing Features ---
    'Illness Yes',
    # --- Financial Insecurity Features ---
    'Uses Foodbank',
    'Worry About Housing Payment',
    'Worry About Energy Payment',
    'Food Insecurity',
    #'Energy Insecurity',

    
    # --- Social Support Features ---
    'Unable to Turn to Friends or Family',
    # --- MSOA/LSOA IMD Derived Features ---
    'IMD Score',
    'Living Environment Score',
    'Population Density',
    
    # --- Geographic and Infrastructure Features ---
    'Distance to Bus Stop',
    'Distance to Underground Access',
    'Distance to Railway Access',
    'Distance to Greenspace',
    
    # --- Loneliness Indicator ---
    'new_loneliness_binary'
]]
model_df_no_food_or_energy_insecurity = model_df.loc[:, [
    # --- Demographic Features ---
    'Age',
    'Income',
    'Unemployed',
    'Not Working (Looking After Child or House)',
    'Not Working Sick or Disabled',
    'Living On my Own With Children Under 18',
    'Gender',
    # --- Health and Wellbeing Features ---
    'Illness Yes',
    # --- Financial Insecurity Features ---
    'Uses Foodbank',
    'Worry About Housing Payment',
    'Worry About Energy Payment',
    #'Food Insecurity',
    #'Energy Insecurity',
    
    # --- Social Support Features ---
    'Unable to Turn to Friends or Family',
    
    # --- MSOA/LSOA IMD Derived Features ---
    'IMD Score',
    'Living Environment Score',
    'Population Density',
    
    # --- Geographic and Infrastructure Features ---
    'Distance to Bus Stop',
    'Distance to Underground Access',
    'Distance to Railway Access',
    'Distance to Greenspace',
    
    # --- Loneliness Indicator ---
    'new_loneliness_binary'
]]

# Creating random train and test sets

In [3]:
random_states=[40,41,42,43,44]
def create_test_train_sets(random_state, dataset_to_split, test_size=0.30, sample_size=500, target_feature='new_loneliness_binary'):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    sets= []

    for i in random_state:
        # Sample 500 observations where new_loneliness_binary is 1
        df_lonely = dataset_to_split[dataset_to_split['new_loneliness_binary'] == 1].sample(n=sample_size, random_state=i)

        # Sample 500 observations where new_loneliness_binary is 0
        df_not_lonely = dataset_to_split[dataset_to_split['new_loneliness_binary'] == 0].sample(n=sample_size, random_state=i)
        # Concatenate the two samples to create model_df
        model_df = pd.concat([df_lonely, df_not_lonely], axis=0).reset_index(drop=True)

        # removes all of the features in the list of features to remove to understand the impact of removing all of the features from the model
        df_copy = model_df.copy()
        X = df_copy.drop(columns=[target_feature])
        y = df_copy[target_feature]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=i, stratify=y)

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
        X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

        # Check for NaNs in the scaled data
        #nan_count_train = X_train_scaled_df.isna().sum().sum()
        #nan_count_test = X_test_scaled_df.isna().sum().sum()
        #if nan_count_train > 0 or nan_count_test > 0:
        #    print(f"Number of NaNs in X_train_scaled_df: {nan_count_train}")
        #    print(f"Number of NaNs in X_test_scaled_df: {nan_count_test}")

        number_of_lonely_in_test = y_test.sum()
        sets.append({
            'X_train': X_train_scaled_df,
            'X_test': X_test_scaled_df,
            'y_train': y_train,
            'y_test': y_test,
            'test_size': test_size,
            'number_of_lonely_in_test': number_of_lonely_in_test
        })
    return sets
def extract_pattern(input_list, start=0, step=4, group_size=4):
    result = []
    for i in range(start, len(input_list), step):
        if i + group_size <= len(input_list):
            result.extend(input_list[i:i+group_size])
    return result
def create_rf_results_df(model_results_list):
    columns = ['Random State', 'Model Accuracy', 'Cross Validation Accuracy', 'Optimal Parameters', 'True Positives',	'False Negatives',	'False Positives',	'True Negatives']
    extracted_elements = extract_pattern(model_results_list)
    rf_result_df = pd.DataFrame([extracted_elements[i:i+8] for i in range(0, len(extracted_elements), 8)], columns=columns)
    return rf_result_df
def dispay_df(df, name, accuracy_column='Model Accuracy', cross_val_column='Cross Validation Accuracy'):
    print(f"\n--- {name} ---")
    print(f"Average Model Accuracy: {df[accuracy_column].mean():.4f}")
    print("\n")
    print(f"Average Cross Validation Accuracy: {df[cross_val_column].mean():.4f}")
    display(df)

In [12]:
df_train_test_sets = create_test_train_sets(random_state=random_states, dataset_to_split=df, sample_size=500, target_feature='new_loneliness_binary')
df_excluding_food_train_test_sets = create_test_train_sets(random_state=random_states, dataset_to_split=df_no_food_insecurity, sample_size=500, target_feature='new_loneliness_binary')
df_excluding_energy_train_test_sets = create_test_train_sets(random_state=random_states, dataset_to_split=df_no_energy_insecurity, sample_size=500, target_feature='new_loneliness_binary')
df_excluding_food_and_energy_train_test_sets = create_test_train_sets(random_state=random_states, dataset_to_split=df_no_food_or_energy_insecurity, sample_size=500, target_feature='new_loneliness_binary')

## Checking the shapes of the generated data

In [14]:
shapes_df = pd.DataFrame(columns=['Set', 'X_train_shape', 'X_test_shape', 'y_train_shape', 'y_test_shape', 'test_size', 'number_of_lonely_in_test'])

for idx, data in enumerate(df_train_test_sets):
    shapes_df = shapes_df.append({
        'Set': idx,
        'X_train_shape': data['X_train'].shape,
        'X_test_shape': data['X_test'].shape,
        'y_train_shape': data['y_train'].shape,
        'y_test_shape': data['y_test'].shape,
        'test_size': data['test_size'],
        'number_of_lonely_in_test': data['number_of_lonely_in_test']
    }, ignore_index=True)

shapes_df

,Set,X_train_shape,X_test_shape,y_train_shape,y_test_shape,test_size,number_of_lonely_in_test
0,0,"(700, 21)","(300, 21)","(700,)","(300,)",0.3,150.0
1,1,"(700, 21)","(300, 21)","(700,)","(300,)",0.3,150.0
2,2,"(700, 21)","(300, 21)","(700,)","(300,)",0.3,150.0
3,3,"(700, 21)","(300, 21)","(700,)","(300,)",0.3,150.0
4,4,"(700, 21)","(300, 21)","(700,)","(300,)",0.3,150.0


In [9]:
shapes_df_2 = pd.DataFrame(columns=['Set', 'X_train_shape', 'X_test_shape', 'y_train_shape', 'y_test_shape', 'test_size', 'number_of_lonely_in_test'])

for idx, data in enumerate(df_train_test_sets):
    shapes_df_2 = shapes_df_2.append({
        'Set': idx,
        'X_train_shape': data['X_train'].shape,
        'X_test_shape': data['X_test'].shape,
        'y_train_shape': data['y_train'].shape,
        'y_test_shape': data['y_test'].shape,
        'test_size': data['test_size'],
        'number_of_lonely_in_test': data['number_of_lonely_in_test']
    }, ignore_index=True)

shapes_df_2

,Set,X_train_shape,X_test_shape,y_train_shape,y_test_shape,test_size,number_of_lonely_in_test
0,0,"(700, 20)","(300, 20)","(700,)","(300,)",0.3,150.0
1,1,"(700, 20)","(300, 20)","(700,)","(300,)",0.3,150.0
2,2,"(700, 20)","(300, 20)","(700,)","(300,)",0.3,150.0
3,3,"(700, 20)","(300, 20)","(700,)","(300,)",0.3,150.0
4,4,"(700, 20)","(300, 20)","(700,)","(300,)",0.3,150.0


# Dummy classifier

In [43]:
def dummy_classifier_loop(data, features_to_remove=None, random_state=random_states):

    for random_state in random_states:
        for i, dataset in enumerate(df_train_test_sets): 
            dummy_clf = DummyClassifier(strategy="prior", random_state=random_state)
            dummy_clf.fit(dataset['X_train'], dataset['y_train'])
            dummy_predictions = dummy_clf.predict(dataset['X_test'])
            #print(f"-------Dataset{i}-------")
            #print(f"Random State: {random_state}")
            a_score = accuracy_score(dataset['y_test'], dummy_predictions)
            
            #print("accuracy score ---- ", accuracy_score(dataset['y_test'], dummy_predictions))
            #print("\n") 
            yield a_score, random_state, train_test_sets

## Dummy Classifier Results

In [44]:
dummy_scores_list = []
dummy_data_split = []
for a_score, random_state, train_test_sets in dummy_classifier_loop(df_train_test_sets):
    dummy_scores_list.append(a_score),
    dummy_scores_list.append(random_state)
    dummy_data_split.append(train_test_sets)
extracted_elements = extract_pattern(dummy_scores_list)
columns = ['model accuracy', 'random_state']
dummy_data_result_df = pd.DataFrame([extracted_elements[i:i+2] for i in range(0, len(extracted_elements), 2)], columns=columns)
dummy_data_result_df

In [151]:
dummy_data_result_df.to_clipboard()

In [47]:
average_accuracy = dummy_data_result_df['model accuracy'].mean()
print(f"Average Model Accuracy: {average_accuracy:.4f}")

Average Model Accuracy: 0.5000


# Logistic Regression 

### Single Logistic Regression Run

In [134]:
from sklearn.linear_model import LogisticRegression
# Create and train the logistic regression model
lineral_model = LogisticRegression(random_state=random_states[0])
# Define the parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 300]
}

# Create the grid search
lr_grid_search = GridSearchCV(lineral_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)

# Fit the grid search

lr_grid_search.fit(df_train_test_sets[0]['X_train'], df_train_test_sets[0]['y_train'])

# Print the best parameters and score
print("Best parameters:", lr_grid_search.best_params_)
print("Best cross-validation score:", lr_grid_search.best_score_)

# Get the best model
best_model = lr_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_model.predict(df_train_test_sets[0]['X_test'])

print("\nClassification Report:")
print(classification_report(df_train_test_sets[0]['y_test'], y_pred))
lr_accuracy = accuracy_score(df_train_test_sets[0]['y_test'], y_pred)
print(f"Model Accuracy: {lr_accuracy:.4f}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best parameters: {'C': 0.1, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Best cross-validation score: 0.7114285714285715

Classification Report:
              precision    recall  f1-score   support

         0.0       0.69      0.69      0.69       150
         1.0       0.69      0.69      0.69       150

    accuracy                           0.69       300
   macro avg       0.69      0.69      0.69       300
weighted avg       0.69      0.69      0.69       300

Model Accuracy: 0.6933


## Logistic Regression Loop

In [9]:
from sklearn.linear_model import LogisticRegression
def LR_loop(custom_test_train_sets, random_state=random_states):
     
    param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 300]
    }

    for random_state in random_states:
        for i, dataset in enumerate(custom_test_train_sets): 
            lineral_model = LogisticRegression(random_state=random_states[i])
                    # Create the grid search
            lr_grid_search = GridSearchCV(lineral_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
            # Fit the grid search
            lr_grid_search.fit(dataset['X_train'], dataset['y_train'])
            # Print the best parameters
            #print("Best parameters found: ", lr_grid_search.best_params_)
            # Get the best model
            best_lr = lr_grid_search.best_estimator_
            cross_validation_accuracy = lr_grid_search.best_score_
            # Make predictions using the best model
            y_pred = best_lr.predict(dataset['X_test'])
            #print(f"-------Dataset{i}-------")
            #print(f"Random State: {random_state}")
            a_score = accuracy_score(dataset['y_test'], y_pred)
            #print("accuracy score ---- ", accuracy_score(dataset['y_test'], y_pred))
            #print("\n") 
            yield a_score, random_state, dataset, best_lr, cross_validation_accuracy

            
lr_scores_list_all_covariates = []
lr_data_split_all_covariates = []
for a_score, random_state, dataset, best_lr, cross_validation_accuracy in LR_loop(custom_test_train_sets=df_train_test_sets):
    lr_scores_list_all_covariates.append(random_state),
    lr_scores_list_all_covariates.append(a_score),
    lr_scores_list_all_covariates.append(cross_validation_accuracy),
    lr_scores_list_all_covariates.append(best_lr),
    lr_data_split_all_covariates.append(dataset)

lr_scores_list_excluding_food = []
lr_data_split_excluding_food = []
for a_score, random_state, dataset, best_lr, cross_validation_accuracy in LR_loop(custom_test_train_sets=df_excluding_food_train_test_sets):
    lr_scores_list_excluding_food.append(random_state),
    lr_scores_list_excluding_food.append(a_score),
    lr_scores_list_excluding_food.append(cross_validation_accuracy),
    lr_scores_list_excluding_food.append(best_lr),
    lr_data_split_excluding_food.append(dataset)

lr_scores_list_excluding_energy = []
lr_data_split_excluding_energy = []
for a_score, random_state, dataset, best_lr, cross_validation_accuracy in LR_loop(custom_test_train_sets=df_excluding_energy_train_test_sets):
    lr_scores_list_excluding_energy.append(random_state),
    lr_scores_list_excluding_energy.append(a_score),
    lr_scores_list_excluding_energy.append(cross_validation_accuracy),
    lr_scores_list_excluding_energy.append(best_lr),
    lr_data_split_excluding_energy.append(dataset)

lr_scores_list_excluding_food_and_energy = []
lr_data_split_excluding_food_and_energy = []
for a_score, random_state, dataset, best_lr, cross_validation_accuracy in LR_loop(custom_test_train_sets=df_excluding_food_and_energy_train_test_sets):
    lr_scores_list_excluding_food_and_energy.append(random_state),
    lr_scores_list_excluding_food_and_energy.append(a_score),
    lr_scores_list_excluding_food_and_energy.append(cross_validation_accuracy),
    lr_scores_list_excluding_food_and_energy.append(best_lr),
    lr_data_split_excluding_food_and_energy.append(dataset)

NameError: name 'income_and_energy_adjusted_df_train_test_sets' is not defined

In [239]:
def create_lr_results_df(model_results_list):
    columns = ['Random State', 'Model Accuracy', 'Cross Validation Accuracy', 'Optimal Parameters']
    extracted_elements = extract_pattern(model_results_list)
    lr_result_df = pd.DataFrame([extracted_elements[i:i+4] for i in range(0, len(extracted_elements), 4)], columns=columns)
    return lr_result_df
def dispay_df(df, name, accuracy_column='Model Accuracy', cross_val_column='Cross Validation Accuracy'):
    print(f"\n--- {name} ---")
    print(f"Average Model Accuracy: {df[accuracy_column].mean():.4f}")
    print("\n")
    print(f"Average Cross Validation Accuracy: {df[cross_val_column].mean():.4f}")
    display(df)

lr_results_all_covariates_df = create_lr_results_df(lr_scores_list_all_covariates)
dispay_df(lr_results_all_covariates_df, "All Covariates")
lr_results_excluding_food_df = create_lr_results_df(lr_scores_list_excluding_food)
dispay_df(lr_results_excluding_food_df, "Excluding Food Insecurity")
lr_results_excluding_energ_df = create_lr_results_df(lr_scores_list_excluding_energy)
dispay_df(lr_results_excluding_energ_df, "Excluding Energy Insecurity")
lr_results_excluding_food_and_energy_df = create_lr_results_df(lr_scores_list_excluding_food_and_energy)
dispay_df(lr_results_excluding_food_and_energy_df, "Excluding Food and Energy Insecurity")


--- All Covariates ---
Average Model Accuracy: 0.7067


Average Cross Validation Accuracy: 0.6929


,model accuracy,random_state,best_LR,cross_validation_accuracy
0,0.723333,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.704286
1,0.720000,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.707143
2,0.696667,40,"LogisticRegression(C=10, random_state=42, solv...",0.684286
3,0.736667,40,"LogisticRegression(C=1, random_state=43, solve...",0.685714
4,0.656667,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.682857
5,0.723333,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.704286
6,0.720000,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.707143
7,0.696667,41,"LogisticRegression(C=10, random_state=42, solv...",0.684286
8,0.736667,41,"LogisticRegression(C=1, random_state=43, solve...",0.685714
9,0.656667,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.682857



--- Excluding Food Insecurity ---
Average Model Accuracy: 0.6907


Average Cross Validation Accuracy: 0.6871


,model accuracy,random_state,best_LR,cross_validation_accuracy
0,0.700000,40,"LogisticRegression(C=1, penalty='l1', random_s...",0.704286
1,0.693333,40,"LogisticRegression(C=1, penalty='l1', random_s...",0.702857
2,0.710000,40,"LogisticRegression(C=1, penalty='l1', random_s...",0.672857
3,0.723333,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.684286
4,0.626667,40,"LogisticRegression(C=0.1, random_state=44, sol...",0.671429
5,0.700000,41,"LogisticRegression(C=1, penalty='l1', random_s...",0.704286
6,0.693333,41,"LogisticRegression(C=1, penalty='l1', random_s...",0.702857
7,0.710000,41,"LogisticRegression(C=1, penalty='l1', random_s...",0.672857
8,0.723333,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.684286
9,0.626667,41,"LogisticRegression(C=0.1, random_state=44, sol...",0.671429



--- Excluding Energy Insecurity ---
Average Model Accuracy: 0.7047


Average Cross Validation Accuracy: 0.6906


,model accuracy,random_state,best_LR,cross_validation_accuracy
0,0.733333,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.701429
1,0.720000,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.707143
2,0.703333,40,"LogisticRegression(C=1, random_state=42, solve...",0.681429
3,0.710000,40,"LogisticRegression(C=0.1, random_state=43, sol...",0.680000
4,0.656667,40,"LogisticRegression(C=0.1, penalty='l1', random...",0.682857
5,0.733333,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.701429
6,0.720000,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.707143
7,0.703333,41,"LogisticRegression(C=1, random_state=42, solve...",0.681429
8,0.710000,41,"LogisticRegression(C=0.1, random_state=43, sol...",0.680000
9,0.656667,41,"LogisticRegression(C=0.1, penalty='l1', random...",0.682857



--- Excluding Food and Energy Insecurity ---
Average Model Accuracy: 0.6833


Average Cross Validation Accuracy: 0.6854


,model accuracy,random_state,best_LR,cross_validation_accuracy
0,0.690000,40,"LogisticRegression(C=1, penalty='l1', random_s...",0.701429
1,0.696667,40,"LogisticRegression(C=1, penalty='l1', random_s...",0.707143
2,0.713333,40,"LogisticRegression(C=1, random_state=42, solve...",0.674286
3,0.710000,40,"LogisticRegression(C=10, random_state=43, solv...",0.674286
4,0.606667,40,"LogisticRegression(C=1, random_state=44, solve...",0.670000
5,0.690000,41,"LogisticRegression(C=1, penalty='l1', random_s...",0.701429
6,0.696667,41,"LogisticRegression(C=1, penalty='l1', random_s...",0.707143
7,0.713333,41,"LogisticRegression(C=1, random_state=42, solve...",0.674286
8,0.710000,41,"LogisticRegression(C=10, random_state=43, solv...",0.674286
9,0.606667,41,"LogisticRegression(C=1, random_state=44, solve...",0.670000


# XGBoost 

### XGBoost Single Run

In [140]:
from xgboost import XGBClassifier 
# Define the parameter grid
param_grid = {
    'n_estimators': [500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'gamma': [0, 0.1, 0.2],
    'reg_lambda': [0, 0.1, 1],
    'scale_pos_weight': [1, 3, 5]
}
xgb_model = XGBClassifier(random_state=random_states[0])

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',  # or another appropriate metric
    n_jobs=-1,
    verbose=2
)

# Assuming X_train and y_train are your training data
grid_search.fit(income_adjusted_train_test_sets[0]['X_train'], df_train_test_sets[0]['y_train'])
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

# Get the best parameters
best_params = grid_search.best_params_

# Create a new model with the best parameters
best_model = XGBClassifier(**best_params, random_state=random_states[0])

# Train the model on the entire training set
best_model.fit(df_train_test_sets[0]['X_train'], df_train_test_sets[0]['y_train'])

# Make predictions on the test set
y_pred = best_model.predict(df_train_test_sets[0]['X_test'])

# Calculate the accuracy
accuracy = accuracy_score(df_train_test_sets[0]['y_test'], y_pred)

print(f"\nAccuracy on test set: {accuracy:.4f}")


Fitting 5 folds for each of 432 candidates, totalling 2160 fits


Best parameters found:  {'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'reg_lambda': 0, 'scale_pos_weight': 1}
Best cross-validation score:  0.6971428571428572

Accuracy on test set: 0.6667


### XGBoost Loop

In [13]:
def XGB_loop(custom_test_train_sets, features_to_remove=None, random_state=random_states):
    from xgboost import XGBClassifier  
    param_grid = {
        'n_estimators': [500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'gamma': [0, 0.1, 0.2],
        'reg_lambda': [0, 0.1, 1],
        'scale_pos_weight': [1, 3, 5]
    }

    for random_state in random_states:
        for i, dataset in enumerate(custom_test_train_sets): 
        
            xgb_model = XGBClassifier(random_state=random_states[i])

    # Instantiate the grid search model
            grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, 
                                    cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

            # Fit the grid search to the data
            grid_search.fit(dataset['X_train'], dataset['y_train'])

            # Print the best parameters
            print("Best parameters found: ", grid_search.best_params_)

            # Get the best model
            best_xgb = grid_search.best_estimator_
            cross_validation_accuracy = grid_search.best_score_
            # Make predictions using the best model
            y_pred = best_xgb.predict(dataset['X_test'])
            
            
            #print(f"-------Dataset{i}-------")
            #print(f"Random State: {random_state}")
            a_score = accuracy_score(dataset['y_test'], y_pred)
            
            print("accuracy score ---- ", accuracy_score(dataset['y_test'], y_pred))
            print("\n") 

            conf_matrix = confusion_matrix(dataset['y_test'], y_pred)
            #print("Confusion Matrix:\n", conf_matrix)

            # Extract TP and FN from confusion matrix
            # Assuming binary classification with labels 0 and 1
            TP = conf_matrix[1, 1]
            FN = conf_matrix[1, 0]
            FP = conf_matrix[0, 1]
            TN = conf_matrix[0, 0]


            yield a_score, random_state, dataset, best_xgb, cross_validation_accuracy, TP, FN,  FP, TN

In [14]:
xgb_scores_list_all_covariates = []
for i, (a_score, random_state, dataset, best_xgb, cross_validation_accuracy, TP, FN, FP, TN) in enumerate(XGB_loop(custom_test_train_sets=df_train_test_sets)):
    xgb_scores_list_all_covariates.extend([
        random_state,
        a_score,
        cross_validation_accuracy,
        best_xgb,
        TP,
        FN,
        FP,
        TN
    ])
xgb_scores_list_all_excluding_food = []
for i, (a_score, random_state, dataset, best_xgb, cross_validation_accuracy, TP, FN, FP, TN) in enumerate(XGB_loop(custom_test_train_sets=df_excluding_food_train_test_sets)):
    xgb_scores_list_all_excluding_food.extend([
        random_state,
        a_score,
        cross_validation_accuracy,
        best_xgb,
        TP,
        FN,
        FP,
        TN
    ])
xgb_scores_list_all_excluding_energy = []
for i, (a_score, random_state, dataset, best_xgb, cross_validation_accuracy, TP, FN, FP, TN) in enumerate(XGB_loop(custom_test_train_sets=df_excluding_energy_train_test_sets)):
    xgb_scores_list_all_excluding_energy.extend([
        random_state,
        a_score,
        cross_validation_accuracy,
        best_xgb,
        TP,
        FN,
        FP,
        TN
    ])
xgb_scores_list_all_excluding_food_and_energy = []
for i, (a_score, random_state, dataset, best_xgb, cross_validation_accuracy, TP, FN, FP, TN) in enumerate(XGB_loop(custom_test_train_sets=df_excluding_food_and_energy_train_test_sets)):
    xgb_scores_list_all_excluding_food_and_energy.extend([
        random_state,
        a_score,
        cross_validation_accuracy,
        best_xgb,
        TP,
        FN,
        FP,
        TN
    ])


Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best parameters found:  {'gamma': 0.1, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'reg_lambda': 1, 'scale_pos_weight': 1}
accuracy score ----  0.6933333333333334


--------- <class 'dict'>
Model 0 saved successfully.
X_train Dataset 0 saved successfully.
X_test Dataset 0 saved successfully.
y_train Dataset 0 saved successfully.
y_test Dataset 0 saved successfully.
Skipping test_size: Value is a float
Skipping number_of_lonely_in_test: Value is a float
Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best parameters found:  {'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 500, 'reg_lambda': 1, 'scale_pos_weight': 1}
accuracy score ----  0.7233333333333334


--------- <class 'dict'>
Model 1 saved successfully.
X_train Dataset 1 saved successfully.
X_test Dataset 1 saved successfully.
y_train Dataset 1 saved successfully.
y_test Dataset 1 saved successfully.
Skipping test_siz

In [16]:

def create_xgb_results_df(model_results_list):
    columns = ['Random State', 'Model Accuracy', 'Cross Validation Accuracy', 'Optimal Parameters', 'True Positives',	'False Negatives',	'False Positives',	'True Negatives']
    extracted_elements = extract_pattern(model_results_list)
    rf_result_df = pd.DataFrame([extracted_elements[i:i+8] for i in range(0, len(extracted_elements), 8)], columns=columns)
    return rf_result_df

def dispay_df(df, name, accuracy_column='Model Accuracy', cross_val_column='Cross Validation Accuracy'):
    print(f"\n--- {name} ---")
    print(f"Average Model Accuracy: {df[accuracy_column].mean():.4f}")
    print("\n")
    print(f"Average Cross Validation Accuracy: {df[cross_val_column].mean():.4f}")
    display(df)

xgb_results_all_covariates_df = create_xgb_results_df(xgb_scores_list_all_covariates)
dispay_df(xgb_results_all_covariates_df, "All Covariates")
xgb_results_excluding_food_df = create_xgb_results_df(xgb_scores_list_all_excluding_food)
dispay_df(xgb_results_excluding_food_df, "Excluding Food Insecurity")
xgb_results_excluding_energ_df = create_xgb_results_df(xgb_scores_list_all_excluding_energy)
dispay_df(xgb_results_excluding_energ_df, "Excluding Energy Insecurity")
xgb_results_excluding_food_and_energy_df = create_xgb_results_df(xgb_scores_list_all_excluding_food_and_energy)
dispay_df(xgb_results_excluding_food_and_energy_df, "Excluding Food and Energy Insecurity")



--- All Covariates ---
Average Model Accuracy: 0.6980


Average Cross Validation Accuracy: 0.6637


,Random State,Model Accuracy,Cross Validation Accuracy,Optimal Parameters,True Positives,False Negatives,False Positives,True Negatives
0,40,0.693333,0.707143,"XGBClassifier(base_score=None, booster=None, c...",106,44,48,102
1,40,0.723333,0.688571,"XGBClassifier(base_score=None, booster=None, c...",108,42,41,109
2,40,0.736667,0.624286,"XGBClassifier(base_score=None, booster=None, c...",111,39,40,110
3,40,0.693333,0.661429,"XGBClassifier(base_score=None, booster=None, c...",109,41,51,99
4,40,0.643333,0.637143,"XGBClassifier(base_score=None, booster=None, c...",104,46,61,89
5,41,0.693333,0.707143,"XGBClassifier(base_score=None, booster=None, c...",106,44,48,102
6,41,0.723333,0.688571,"XGBClassifier(base_score=None, booster=None, c...",108,42,41,109
7,41,0.736667,0.624286,"XGBClassifier(base_score=None, booster=None, c...",111,39,40,110
8,41,0.693333,0.661429,"XGBClassifier(base_score=None, booster=None, c...",109,41,51,99
9,41,0.643333,0.637143,"XGBClassifier(base_score=None, booster=None, c...",104,46,61,89



--- Excluding Food Insecurity ---
Average Model Accuracy: 0.6613


Average Cross Validation Accuracy: 0.6646


,Random State,Model Accuracy,Cross Validation Accuracy,Optimal Parameters,True Positives,False Negatives,False Positives,True Negatives
0,40,0.696667,0.715714,"XGBClassifier(base_score=None, booster=None, c...",108,42,49,101
1,40,0.660000,0.678571,"XGBClassifier(base_score=None, booster=None, c...",95,55,47,103
2,40,0.686667,0.625714,"XGBClassifier(base_score=None, booster=None, c...",100,50,44,106
3,40,0.646667,0.661429,"XGBClassifier(base_score=None, booster=None, c...",131,19,87,63
4,40,0.616667,0.641429,"XGBClassifier(base_score=None, booster=None, c...",89,61,54,96
5,41,0.696667,0.715714,"XGBClassifier(base_score=None, booster=None, c...",108,42,49,101
6,41,0.660000,0.678571,"XGBClassifier(base_score=None, booster=None, c...",95,55,47,103
7,41,0.686667,0.625714,"XGBClassifier(base_score=None, booster=None, c...",100,50,44,106
8,41,0.646667,0.661429,"XGBClassifier(base_score=None, booster=None, c...",131,19,87,63
9,41,0.616667,0.641429,"XGBClassifier(base_score=None, booster=None, c...",89,61,54,96



--- Excluding Energy Insecurity ---
Average Model Accuracy: 0.6833


Average Cross Validation Accuracy: 0.6620


,Random State,Model Accuracy,Cross Validation Accuracy,Optimal Parameters,True Positives,False Negatives,False Positives,True Negatives
0,40,0.700000,0.701429,"XGBClassifier(base_score=None, booster=None, c...",108,42,48,102
1,40,0.710000,0.688571,"XGBClassifier(base_score=None, booster=None, c...",99,51,36,114
2,40,0.690000,0.624286,"XGBClassifier(base_score=None, booster=None, c...",105,45,48,102
3,40,0.700000,0.655714,"XGBClassifier(base_score=None, booster=None, c...",113,37,53,97
4,40,0.616667,0.640000,"XGBClassifier(base_score=None, booster=None, c...",101,49,66,84
5,41,0.700000,0.701429,"XGBClassifier(base_score=None, booster=None, c...",108,42,48,102
6,41,0.710000,0.688571,"XGBClassifier(base_score=None, booster=None, c...",99,51,36,114
7,41,0.690000,0.624286,"XGBClassifier(base_score=None, booster=None, c...",105,45,48,102
8,41,0.700000,0.655714,"XGBClassifier(base_score=None, booster=None, c...",113,37,53,97
9,41,0.616667,0.640000,"XGBClassifier(base_score=None, booster=None, c...",101,49,66,84



--- Excluding Food and Energy Insecurity ---
Average Model Accuracy: 0.6787


Average Cross Validation Accuracy: 0.6597


,Random State,Model Accuracy,Cross Validation Accuracy,Optimal Parameters,True Positives,False Negatives,False Positives,True Negatives
0,40,0.683333,0.700000,"XGBClassifier(base_score=None, booster=None, c...",107,43,52,98
1,40,0.706667,0.677143,"XGBClassifier(base_score=None, booster=None, c...",108,42,46,104
2,40,0.676667,0.624286,"XGBClassifier(base_score=None, booster=None, c...",97,53,44,106
3,40,0.706667,0.660000,"XGBClassifier(base_score=None, booster=None, c...",118,32,56,94
4,40,0.620000,0.637143,"XGBClassifier(base_score=None, booster=None, c...",107,43,71,79
5,41,0.683333,0.700000,"XGBClassifier(base_score=None, booster=None, c...",107,43,52,98
6,41,0.706667,0.677143,"XGBClassifier(base_score=None, booster=None, c...",108,42,46,104
7,41,0.676667,0.624286,"XGBClassifier(base_score=None, booster=None, c...",97,53,44,106
8,41,0.706667,0.660000,"XGBClassifier(base_score=None, booster=None, c...",118,32,56,94
9,41,0.620000,0.637143,"XGBClassifier(base_score=None, booster=None, c...",107,43,71,79
